In [1]:
# The adult income dataset
# Note: The dataset looks skewed, the number of > 50K samples are less than the number of <=50K ones


In [2]:
import torch
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def convert_low_freq_to_other(df: pd.DataFrame, col_name: str, top: int = 10):
    df[col_name] = df[col_name].astype(str)
    top10 = df[col_name].value_counts().nlargest(top).index
    df[col_name] = df[col_name].where(df[col_name].isin(top10), "Other")
    return df


def get_preprocessed():
    df = fetch_openml("adult", version=2, as_frame=True)['frame']

    y = df['class'].map({'>50K': 1, '<=50K': 0})

    X = df.drop(columns=['fnlwgt', 'education', 'class'])
    X = X.fillna(X.mode().iloc[0])
    X['age'] = X['age'] / X['age'].max()
    X['hours-per-week'] = X['hours-per-week'] / X['hours-per-week'].max()
    X['capital-gain'] = np.log10(X['capital-gain'] + 1) # +1 added to avoid log zero
    X['capital-loss'] = np.log10(X['capital-loss'] + 1)

    X = convert_low_freq_to_other(X, "native-country")

    X = pd.get_dummies(X, columns=['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country'])

    return X, y

def to_tensor(df: pd.DataFrame, cls):
    return torch.from_numpy(df.to_numpy(dtype=cls))

# Utility functions

In [ ]:
def loss_function(predicted: torch.Tensor, actual: torch.Tensor):
    # MSE as loss function
    # return torch.mean(torch.square(predicted - actual))
    # BCE
        return torch.mean(
        -(actual * torch.log(predicted + 1e-8) +
          (1 - actual) * torch.log(1 - predicted + 1e-8))
    )

def accuracy(predicted: torch.Tensor, actual: torch.Tensor):
    total = predicted.numel()
    if total == 0:
        return 0
    correct = (predicted == actual).sum().item()
    accuracy =  correct / total
    return accuracy

def print_metrics(epoch: int, X_train: torch.Tensor, X_valid: torch.Tensor, y_train: torch.Tensor, y_valid: torch.Tensor):
    train_pred = get_prediction(X_train, coeffs, bias) # type: ignore
    valid_pred = get_prediction(X_valid, coeffs, bias) # type: ignore
    print(f"{epoch}: loss(train): {loss_function(train_pred, y_train)}", end=" ")
    print(f"accuracy(train): {accuracy(train_pred, y_train):0.2f}", end=" ")
    print(f"loss(validation): {loss_function(valid_pred, y_valid):0.2f}", end=" ")
    print(f"accuracy(validation): {accuracy(valid_pred, y_valid):0.2f}")

# Simple linear model

In [4]:
torch.manual_seed(42)
np.random.seed(42)

def get_coeffs(X: torch.Tensor):
    torch.manual_seed(42)
    np.random.seed(42)
    coeffs = torch.rand(X.shape[1]) - 0.5
    coeffs.requires_grad_()
    bias = torch.rand(1, requires_grad=True)
    return coeffs, bias

def get_prediction(X: torch.Tensor, coeffs: torch.Tensor, bias: torch.Tensor):
    return (get_prediction_proba(X, coeffs, bias) > 0.5).float()

def get_prediction_proba(X: torch.Tensor, coeffs: torch.Tensor, bias: torch.Tensor):
    return torch.sigmoid((X * coeffs).sum(dim=1) + bias)

In [5]:
X, y = get_preprocessed()
X, y = to_tensor(X, np.float32), to_tensor(y, np.float32)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)
coeffs, bias = get_coeffs(X)

In [6]:
# Check out the initial loss, accuracy (from random coeffs)
print_metrics(0, X_train, X_valid, y_train, y_valid)

0: loss(train): 13.966507911682129 accuracy(train): 0.24 loss(validation): 14.12 accuracy(validation): 0.23


In [7]:
lr = 0.2
epochs = 10

coeffs, bias = get_coeffs(X_train)

for epoch in range(epochs):
    # Full batch gradient descent
    preds = get_prediction_proba(X_train, coeffs, bias)
    loss = loss_function(preds, y_train)
    loss.backward()
    
    with torch.no_grad():
        coeffs -= coeffs.grad * lr # type: ignore
        bias -= bias.grad * lr # type: ignore
        coeffs.grad.zero_() # type: ignore
        bias.grad.zero_() # type: ignore
    print_metrics(epoch, X_train, X_valid, y_train, y_valid)
# Got 0.84 accuracy on validation with 1000 epochs and LR of 0.1

0: loss(train): 4.480036735534668 accuracy(train): 0.76 loss(validation): 4.32 accuracy(validation): 0.77
1: loss(train): 4.685854434967041 accuracy(train): 0.75 loss(validation): 4.51 accuracy(validation): 0.76
2: loss(train): 13.985905647277832 accuracy(train): 0.24 loss(validation): 14.14 accuracy(validation): 0.23
3: loss(train): 4.474110126495361 accuracy(train): 0.76 loss(validation): 4.32 accuracy(validation): 0.77
4: loss(train): 4.659453868865967 accuracy(train): 0.75 loss(validation): 4.49 accuracy(validation): 0.76
5: loss(train): 13.973512649536133 accuracy(train): 0.24 loss(validation): 14.12 accuracy(validation): 0.23
6: loss(train): 4.473571300506592 accuracy(train): 0.76 loss(validation): 4.32 accuracy(validation): 0.77
7: loss(train): 4.6540656089782715 accuracy(train): 0.75 loss(validation): 4.48 accuracy(validation): 0.76
8: loss(train): 13.958965301513672 accuracy(train): 0.24 loss(validation): 14.10 accuracy(validation): 0.23
9: loss(train): 4.470877170562744 accur

# Using matrix multiplication, multiple neurons 

m - number of training examples
n - number of input parameters
l - number of neurons in network

Size of each layer
```
X - m x n
W1 - n x l
b1 - l
W2 - l x 1
b2 - 1
```

First layer, from the input to output of each neuron for a training example,
$$h = ReLU(X \times W_1 + b_1)$$
h is of size $(m \times n) \times (n \times l) = m \times l$

Second layer, from hidden layer to output
$$f = h \times W_2 + b_2$$
f is of size $(m \times l) \times (l \times 1) = m \times 1$

I.e. one ouptut for each training example
$$y = \sigma{(f)}$$
Sigmoid is run to squish the values between 0 and 1

In [8]:
X, y = get_preprocessed()
X, y = to_tensor(X, np.float32), to_tensor(y, np.float32)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)

In [9]:
from typing import Callable

def dense(n_inputs: int, n_outputs: int):
    # Create a matrix of size n_inputs x n_outputs
    # And a bias matrix of size 1 x n_outputs (for pytorch broadcasting)
    # Note: This is a dense layer, and the number of neurons is equal to n_outputs

    W = torch.randn(n_inputs, n_outputs)
    b = torch.randn(n_outputs)
    W.requires_grad_()
    b.requires_grad_()
    return W, b

def forward(X: torch.Tensor, layer: tuple[torch.Tensor, torch.Tensor], activation: Callable[[torch.Tensor], torch.Tensor] = torch.relu):
    W, b = layer
    A = activation(X @ W + b)
    return A

In [61]:
hidden_size = 64
torch.manual_seed(42)
np.random.seed(42)
W1, b1 = dense(X_train.shape[1], hidden_size)
W2, b2 = dense(hidden_size, 1)

# Xavier initialization
W1.data *= np.sqrt(2 / X_train.shape[1])
W2.data *= np.sqrt(2 / hidden_size)

print(W1.shape, b1.shape)
print(W2.shape, b2.shape)
# Note: A mismatch of size (A2 - 34189 x 1, and y_train 34189 was causing huge memory issues, maybe due to broadcasting)
y_train = y_train.view(-1, 1)
y_valid = y_valid.view(-1, 1)

epochs  = 500
lr = 0.1
for epoch in range(epochs):

    # A forward pass
    A1 = forward(X_train, (W1, b1), activation=torch.relu)
    A2 = forward(A1, (W2, b2), activation=torch.sigmoid)
    loss = loss_function(A2, y_train)
    loss.backward()
    
    with torch.no_grad():
        W1 -= lr * W1.grad # type: ignore
        b1 -= lr * b1.grad # type: ignore

        W2 -= lr * W2.grad # type: ignore
        b2 -= lr * b2.grad # type: ignore
        
        W1.grad.zero_() # type: ignore
        b1.grad.zero_() # type: ignore
        W2.grad.zero_() # type: ignore
        b2.grad.zero_() # type: ignore

    if (epoch+1) % 100 == 0:
        print(f"=== Epoch {epoch}===")
        print("train loss", loss.item(), "train accuracy", accuracy((A2 > 0.5), y_train))
        with torch.no_grad():
            # Compute validation metrics
            A1 = forward(X_valid, (W1, b1), activation=torch.relu)
            A2 = forward(A1, (W2, b2), activation=torch.sigmoid)
            loss = loss_function(A2, y_valid)
            print("valid loss", loss.item(), "valid accuracy", accuracy((A2 > 0.5), y_valid))

torch.Size([58, 64]) torch.Size([64])
torch.Size([64, 1]) torch.Size([1])
=== Epoch 99===
train loss 0.39310917258262634 train accuracy 0.8263769048524379
valid loss 0.3825588524341583 valid accuracy 0.8078891694533543
=== Epoch 199===
train loss 0.36183294653892517 train accuracy 0.8330749656322209
valid loss 0.34986457228660583 valid accuracy 0.8377124138401693
=== Epoch 299===
train loss 0.3575778603553772 train accuracy 0.8329872181110883
valid loss 0.34535902738571167 valid accuracy 0.8423531017539071
=== Epoch 399===
train loss 0.35356834530830383 train accuracy 0.8348591652285823
valid loss 0.3418072760105133 valid accuracy 0.8435815191428376
=== Epoch 499===
train loss 0.3518044054508209 train accuracy 0.8354441487027991
valid loss 0.3401753008365631 valid accuracy 0.843786255374326


# Deep learning - multiple layers

In [123]:
from typing import Callable

def loss_fn(predicted: torch.Tensor, actual: torch.Tensor):
    # BCE
    return torch.mean(
        -(actual * torch.log(predicted + 1e-8) +
          (1 - actual) * torch.log(1 - predicted + 1e-8))
    )

def accuracy(predicted: torch.Tensor, actual: torch.Tensor):
    total = predicted.numel()
    if total == 0:
        return 0
    correct = (predicted == actual).sum().item()
    accuracy =  correct / total
    return accuracy


class Dense:
    def __init__(self, n_inputs: int, n_outputs: int, activation: Callable[[torch.Tensor], torch.Tensor] = torch.relu):
        self.n_inputs = n_inputs
        self.n_outputs = n_outputs
        self.activation = activation
        # He initialization
        self.W = torch.randn(self.n_inputs, n_outputs) * torch.sqrt(torch.tensor(2/n_inputs))
        self.b = torch.randn(n_outputs)
        self.W.requires_grad_()
        self.b.requires_grad_()
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        A = self.activation(X @ self.W + self.b)
        return A
    
    def backward(self, lr: float):
        with torch.no_grad():
            if self.W.grad is not None:
                self.W -= lr * self.W.grad
                self.W.grad.detach_() # type: ignore
                self.W.grad.zero_() # type: ignore
            if self.b.grad is not None:
                self.b -= lr * self.b.grad
                self.b.grad.detach_() # type: ignore
                self.b.grad.zero_() # type: ignore

class Sequential:
    def __init__(self, layers: list[Dense]) -> None:
        self.layers = layers
    
    def predict_proba(self, X: torch.Tensor, grad = False):
        if grad:
            for layer in self.layers:
                X = layer.forward(X)
            return X

        with torch.no_grad():
            for layer in self.layers:
                X = layer.forward(X)
        return X.detach()
    
    def predict(self, X: torch.Tensor):
        preds = self.predict_proba(X)
        return self._predict_from_proba(preds)
    
    def _predict_from_proba(self, X: torch.Tensor):
        return (X > 0.5).float()
    
    def fit(self, 
            x_train: torch.Tensor,
            x_valid: torch.Tensor,
            y_train: torch.Tensor,
            y_valid: torch.Tensor,
            lr: float = 0.1,
            epochs: int = 10,
            loss_function: Callable[[torch.Tensor, torch.Tensor], torch.Tensor] = loss_fn,
            print_metrics_every = 1):
        for epoch in range(epochs):
            preds = self.predict_proba(x_train, grad=True)
            loss = loss_function(preds, y_train)
            loss.backward()

            train_loss = loss.detach()
            train_acc = accuracy(self._predict_from_proba(preds), y_train)

            
            for layer in self.layers:
                layer.backward(lr)

            if epoch % print_metrics_every == 0:
                # Validation
                preds = self.predict_proba(x_valid)
                loss = loss_function(preds, y_valid)
                acc = accuracy(self._predict_from_proba(preds), y_valid)
                print(f"{epoch}: train loss: {train_loss:0.2f} acc:{train_acc:0.2f}, valid loss:{loss:0.2f} acc:{acc:0.2f}")

        preds = self.predict_proba(x_valid)
        loss = loss_function(preds, y_valid)
        acc = accuracy(self._predict_from_proba(preds), y_valid)
        print(f"final: validation loss:{loss:0.2f} acc:{acc:0.2f}")

In [129]:
X, y = get_preprocessed()
X, y = to_tensor(X, np.float32), to_tensor(y, np.float32)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)
y_train = y_train.view(-1, 1)
y_valid = y_valid.view(-1, 1)

np.random.seed(42)
torch.manual_seed(42)

network = Sequential([
    Dense(X.shape[1], 64),
    Dense(64, 32),
    Dense(32, 16),
    Dense(16, 1, activation=torch.sigmoid)
])

network.fit(X_train, X_valid,  y_train,  y_valid, lr = 0.01, epochs = 500, print_metrics_every=50)

0: train loss: 0.99 acc:0.76, valid loss:0.65 acc:0.77
50: train loss: 0.46 acc:0.79, valid loss:0.45 acc:0.80
100: train loss: 0.42 acc:0.81, valid loss:0.41 acc:0.82
150: train loss: 0.40 acc:0.82, valid loss:0.39 acc:0.82
200: train loss: 0.38 acc:0.83, valid loss:0.37 acc:0.83
250: train loss: 0.37 acc:0.83, valid loss:0.36 acc:0.83
300: train loss: 0.37 acc:0.83, valid loss:0.36 acc:0.84
350: train loss: 0.36 acc:0.83, valid loss:0.35 acc:0.84
400: train loss: 0.36 acc:0.83, valid loss:0.35 acc:0.84
450: train loss: 0.36 acc:0.83, valid loss:0.35 acc:0.84
final: validation loss:0.35 acc:0.84


In [143]:
# Testing out with XOR

import torch
import numpy as np

X = torch.tensor([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=torch.float32)

y = torch.tensor([
    [0],
    [1],
    [1],
    [0]
], dtype=torch.float32)

network = Sequential([
    Dense(2, 2, activation=torch.tanh),
    Dense(2, 1, activation=torch.sigmoid)
])

network.fit(
    x_train=X,
    x_valid=X,
    y_train=y,
    y_valid=y,
    lr=0.1,
    epochs=500,
    print_metrics_every=50
)

preds = network.predict(X)
print("Predictions:")
for inp, pred in zip(X, preds):
    print(f"Input: {inp.numpy()}, Output: {pred.item():.1f}")

0: train loss: 0.94 acc:0.50, valid loss:0.90 acc:0.50
50: train loss: 0.63 acc:0.75, valid loss:0.62 acc:0.75
100: train loss: 0.58 acc:0.75, valid loss:0.58 acc:0.75
150: train loss: 0.54 acc:0.75, valid loss:0.54 acc:0.75
200: train loss: 0.51 acc:0.75, valid loss:0.51 acc:0.75
250: train loss: 0.47 acc:0.75, valid loss:0.47 acc:0.75
300: train loss: 0.44 acc:0.75, valid loss:0.44 acc:0.75
350: train loss: 0.40 acc:0.75, valid loss:0.40 acc:0.75
400: train loss: 0.35 acc:1.00, valid loss:0.35 acc:1.00
450: train loss: 0.30 acc:1.00, valid loss:0.30 acc:1.00
final: validation loss:0.26 acc:1.00
Predictions:
Input: [0. 0.], Output: 0.0
Input: [0. 1.], Output: 1.0
Input: [1. 0.], Output: 1.0
Input: [1. 1.], Output: 0.0


In [164]:
import random
X, y = get_preprocessed()
X, y = to_tensor(X, np.float32), to_tensor(y, np.float32)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)
y_train = y_train.view(-1, 1)
y_valid = y_valid.view(-1, 1)

def ensemble(n: int = 10):
    networks: list[Sequential] = []
    Hs = [64, 128, 32, 16]
    for i in range(n):
        H = random.choice(Hs)
        network = Sequential([
            Dense(X.shape[1], H),
            Dense(H, H // 2),
            Dense(H // 2, H // 4),
            Dense(H // 4, 1, activation=torch.sigmoid)
        ])
        network.fit(X_train, X_valid,  y_train,  y_valid, lr = 0.01, epochs = 500, print_metrics_every=1000)
        networks.append(network)
    return networks

def ensemble_predict(X: torch.Tensor, ensemble: list[Sequential]):
    s = torch.zeros(X.shape[0], 1)
    for network in ensemble:
        s += network.predict_proba(X)
    return s / len(ensemble)

In [165]:
e = ensemble(5)

0: train loss: 2.73 acc:0.24, valid loss:1.10 acc:0.26
final: validation loss:0.36 acc:0.84
0: train loss: 1.17 acc:0.22, valid loss:0.72 acc:0.70
final: validation loss:0.35 acc:0.84
0: train loss: 1.06 acc:0.76, valid loss:0.81 acc:0.77
final: validation loss:0.36 acc:0.84
0: train loss: 3.08 acc:0.24, valid loss:0.87 acc:0.32
final: validation loss:0.35 acc:0.84
0: train loss: 0.67 acc:0.65, valid loss:0.60 acc:0.76
final: validation loss:0.35 acc:0.84


In [173]:
preds = ensemble_predict(X_valid, e[0:5])
loss = loss_function(preds, y_valid)
acc = accuracy((preds > 0.5).float(), y_valid)
print(f"final: validation loss:{loss:0.3f} acc:{acc:0.5f}")

final: validation loss:0.346 acc:0.84222


In [174]:

X, y = get_preprocessed()
X, y = to_tensor(X, np.float32), to_tensor(y, np.float32)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)
y_train = y_train.view(-1, 1)
y_valid = y_valid.view(-1, 1)

np.random.seed(42)
torch.manual_seed(42)

network = Sequential([
    Dense(X.shape[1], 64),
    Dense(64, 32),
    Dense(32, 16),
    Dense(16, 1, activation=torch.sigmoid)
])

network.fit(X_train, X_valid,  y_train,  y_valid, lr = 0.01, epochs = 500, print_metrics_every=50)

0: train loss: 0.99 acc:0.76, valid loss:0.65 acc:0.77
50: train loss: 0.46 acc:0.79, valid loss:0.45 acc:0.80
100: train loss: 0.42 acc:0.81, valid loss:0.41 acc:0.82
150: train loss: 0.40 acc:0.82, valid loss:0.39 acc:0.82
200: train loss: 0.38 acc:0.83, valid loss:0.37 acc:0.83
250: train loss: 0.37 acc:0.83, valid loss:0.36 acc:0.83
300: train loss: 0.37 acc:0.83, valid loss:0.36 acc:0.84
350: train loss: 0.36 acc:0.83, valid loss:0.35 acc:0.84
400: train loss: 0.36 acc:0.83, valid loss:0.35 acc:0.84
450: train loss: 0.36 acc:0.83, valid loss:0.35 acc:0.84
final: validation loss:0.35 acc:0.84


In [176]:
import torch.nn as nn
import torch
import torch.optim as optim

model = nn.Sequential(
    nn.Linear(X.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)
epochs = 500
print_metrics_every = 50

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    train_logits = model(X_train)
    loss = criterion(train_logits, y_train)
    
    # Backward pass
    loss.backward()
    optimizer.step()

    if epoch % print_metrics_every == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            valid_logits = model(X_valid)
            valid_loss = criterion(valid_logits, y_valid)
            preds = (torch.sigmoid(valid_logits) > 0.5).float()
            acc = (preds == y_valid).float().mean()
            print(f"Epoch {epoch:3}: Train Loss: {loss.item():.4f} | "
                  f"Valid Loss: {valid_loss.item():.4f} | Valid Acc: {acc.item():.4f}")

Epoch   1: Train Loss: 0.6690 | Valid Loss: 0.6345 | Valid Acc: 0.7666
Epoch  50: Train Loss: 0.3399 | Valid Loss: 0.3287 | Valid Acc: 0.8497
Epoch 100: Train Loss: 0.3294 | Valid Loss: 0.3215 | Valid Acc: 0.8497
Epoch 150: Train Loss: 0.3193 | Valid Loss: 0.3184 | Valid Acc: 0.8512
Epoch 200: Train Loss: 0.3081 | Valid Loss: 0.3136 | Valid Acc: 0.8549
Epoch 250: Train Loss: 0.3018 | Valid Loss: 0.3109 | Valid Acc: 0.8566
Epoch 300: Train Loss: 0.2975 | Valid Loss: 0.3162 | Valid Acc: 0.8537
Epoch 350: Train Loss: 0.2902 | Valid Loss: 0.3151 | Valid Acc: 0.8580
Epoch 400: Train Loss: 0.2873 | Valid Loss: 0.3224 | Valid Acc: 0.8538
Epoch 450: Train Loss: 0.2882 | Valid Loss: 0.3228 | Valid Acc: 0.8572
Epoch 500: Train Loss: 0.2878 | Valid Loss: 0.3323 | Valid Acc: 0.8504
